<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "533"
iso_country = "Aruba"
import_portfolio = "h4_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2023
gdp = 3.650*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Aruba': 

Number of trade records present in dataset: 7182
Number of partner countries present in dataset: 10
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$940.3m
Total weight within the dataset: 683.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,533,Aruba,076,Brazil,Import,Sea,401693,401693 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",580.941297,NaN,580.941297,kg,26.64749,26.64749,56.467494,7105.0,4414.840955,249.295005,['Container'],['Container'],1.288476,1.286210,1.286432,1.286432,1.287447,1.287447,1.347928,2.390398,1.485757,1.380522,1.384290,1.418746,1.319805,2.340425,1.564488,1.433447,1.400437,1.464026,1.295141,2.546955,1.940522,1.697254,1.570096,1.526023,1.270933,2.707452,2.227741,2.086346,1.680854,1.635613,1.247182,2.744994,2.623973,2.647368,1.821152,1.718200,1.222955,2.688471,2.574301,2.819511,2.089497,1.749733,1.200132,2.396679,2.499207,2.618462,1.959671,1.770410,1.176859,2.201700,2.237605,2.783733,2.301361,1.78218,3627
1,1,533,Aruba,076,Brazil,Import,Sea,441820,441820 | Wood; doors and thei,4418,4418 | Builders' joinery an,44,44 | Wood and articles of,316,Builders' joinery and carpentry of wood (inclu...,Wood and Paper,30,LUM - Wood products,371131.943000,NaN,371131.943000,kg,168136.90000,168136.90000,159735.188267,7105.0,4414.840955,705205.451116,['all'],['all'],2180.879750,2177.144902,2177.542333,2177.542333,2179.201990,2179.354203,2263.337262,3709.618113,2413.978467,2274.220951,2282.290354,2353.864100,2226.457431,3650.431965,2573.825943,2373.099693,2333.602178,2419.432500,2190.447736,4007.039205,3045.531675,2779.339064,2604.670767,2505.644310,2155.553539,4362.117637,3535.681774,3331.352197,2718.386827,2659.259186,2120.379943,4544.795758,4193.703261,4338.270275,2892.166974,2779.482481,2084.731540,4186.451392,4050.152982,4327.910839,3405.179882,2824.034716,2051.700616,3856.819570,3912.553648,4341.479970,3405.223158,2850.381602,2018.044437,3511.207887,3563.162899,4116.766795,3558.394162,2871.14476,316


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $299.9m (31.9%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
22 | Beverages, spirits a  50745809.1  133352389.2      5.4     19.5
02 | Meat and edible meat  36516605.7   13227705.2      3.9      1.9
04 | Dairy produce; birds  33602802.5   13864345.1      3.6      2.0
19 | Preparations of cere  25060715.0    9252585.0      2.7      1.4
21 | Miscellaneous edible  22639088.5    7679168.7      2.4      1.1
07 | Vegetables and certa  22204792.8   26500947.1      2.4      3.9
16 | Meat, fish, crustace  19292812.3    4293271.9      2.1      0.6
20 | Preparations of vege  18540030.9   11792161.5      2.0      1.7
08 | Fruit and nuts, edib  17850527.7   15268383.9      1.9      2.2
03 | Fish and crustaceans  11965144.6    1958132.8      1.3      0.3
23 | Food industries, res  10574885.5    7830143.6      1.1      1.1
15

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $306.3m (32.6%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  183145646.1  178620864.9     19.5     26.1
Agriculture       118440314.8   82115856.1     12.6     12.0
Fishing             4694834.3     809352.4      0.5      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $227.3m (24.2%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OFD - Food products n.e.c.     100688464.7  53418492.2     10.7      7.8
OMT - Meat products n.e.c.      35520882.6  13358812.1      3.8      2.0
V_F - Vegetables, fruit, nuts   32168674.6  35366345.0      3.4      5.2
MIL - Dairy products            29681182.9  10084291.6      3.2      1.5
CMT - Bovine meat prods         17846775.0   3607932.5      1.9      0.5
VOL - Vegetable oils and fats    4254182.2   2473284.4      0.5      0.4
OCR - Crops n.e.c.               2915766.4   1855856.3      0.3      0.3
PCR - Processed rice             1514392.5   2630499.7      0.2      0.4
PDR - Paddy rice                 1121078.8   1947313.9      0.1      0.3
FSH - Fishing                     496391.1     95314.4      0.1      0.0
SGR - Sugar                       469893.7   1082655.6    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 31.9% 
GTAP 24.2% 
EORA 32.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23060/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,533,Aruba,076,Brazil,Import,Sea,401693,401693 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",580.941297,NaN,580.941297,kg,26.64749,26.64749,56.467494,7105.0,4414.840955,249.295005,['Container'],['Container'],1.288476,1.286210,1.286432,1.286432,1.287447,1.287447,1.347928,2.390398,1.485757,1.380522,1.384290,1.418746,1.319805,2.340425,1.564488,1.433447,1.400437,1.464026,1.295141,2.546955,1.940522,1.697254,1.570096,1.526023,1.270933,2.707452,2.227741,2.086346,1.680854,1.635613,1.247182,2.744994,2.623973,2.647368,1.821152,1.718200,1.222955,2.688471,2.574301,2.819511,2.089497,1.749733,1.200132,2.396679,2.499207,2.618462,1.959671,1.770410,1.176859,2.201700,2.237605,2.783733,2.301361,1.78218,3627,NaN
1,1,533,Aruba,076,Brazil,Import,Sea,441820,441820 | Wood; doors and thei,4418,4418 | Builders' joinery an,44,44 | Wood and articles of,316,Builders' joinery and carpentry of wood (inclu...,Wood and Paper,30,LUM - Wood products,371131.943000,NaN,371131.943000,kg,168136.90000,168136.90000,159735.188267,7105.0,4414.840955,705205.451116,['all'],['all'],2180.879750,2177.144902,2177.542333,2177.542333,2179.201990,2179.354203,2263.337262,3709.618113,2413.978467,2274.220951,2282.290354,2353.864100,2226.457431,3650.431965,2573.825943,2373.099693,2333.602178,2419.432500,2190.447736,4007.039205,3045.531675,2779.339064,2604.670767,2505.644310,2155.553539,4362.117637,3535.681774,3331.352197,2718.386827,2659.259186,2120.379943,4544.795758,4193.703261,4338.270275,2892.166974,2779.482481,2084.731540,4186.451392,4050.152982,4327.910839,3405.179882,2824.034716,2051.700616,3856.819570,3912.553648,4341.479970,3405.223158,2850.381602,2018.044437,3511.207887,3563.162899,4116.766795,3558.394162,2871.14476,316,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $165.0m of $940.3m (17.6%)
Volume: 109.3 of 683.3 million kilograms (16.0%)
Value-to-Weight: 1.51 from 1.38 US$ per kg (+9.8%)
Average distance to market (volume-weighted): 2268.0 from 1995.7 nm (+13.6%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    179  57400062.1  31715354.3      6.1      4.6
1                Dairy     44  29223204.9  11581073.7      3.1      1.7
2           Vegetables    133  26599418.1  30137152.3      2.8      4.4
3        Meat and Prep     44  21702891.9  10019189.3      2.3      1.5
4               Fruits     97  16650768.0  14728318.2      1.8      2.2
5        Fats and Oils     44   6658956.8   4740698.3      0.7      0.7
6      Sugar and Honey     37   6580104.0   5523753.5      0.7      0.8
7  Unprocessed Cereals     14    216380.9    824003.0      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,15716137.9,27738318.4,17287862.7,16077027.2,16121746.5,16529496.3,14552794.8,31988065.5,30508002.1,30825480.8,21160512.9,19988314.0,13739515.8,25590429.2,26004918.0,32210937.9,26703414.2,20727347.8
1,Fats and Oils,4855527.1,8501328.2,5326139.9,4956363.7,4970014.0,5100529.6,4502719.7,9753886.7,9340641.1,9461552.1,6506104.0,6146171.5,4256255.4,7852452.9,7976494.2,9830562.1,8149065.5,6369228.7
2,Fruits,11875504.8,21027522.9,13081229.6,12158004.4,12191383.5,12496408.4,10990708.2,24176648.5,23094995.0,23312304.5,16027094.6,15126424.0,10372829.4,19377609.4,19693045.9,24465960.2,20242784.3,15688416.7
3,Meat and Prep,17374140.3,30811082.4,19150697.0,17794270.7,17842837.0,18286952.9,16075580.5,35381656.0,33821751.5,34123303.2,23473774.3,22146770.7,15169152.5,28378858.4,28841658.1,35880986.4,29663438.1,22971442.1
4,Processed Cereals,36508457.7,63439035.0,39926165.9,37222447.2,37326853.2,38311914.9,33899880.0,73545105.0,70086753.6,71017567.8,48962427.4,46027984.3,32076082.3,58981507.7,59929310.7,73509792.8,61113827.2,47665170.2
5,Sugar and Honey,4266914.8,7334163.6,4644871.3,4338502.2,4351190.2,4470122.2,3968906.6,8577312.4,8133704.8,8272141.1,5674823.1,5351966.1,3759826.3,6841835.3,6950136.5,8438435.2,7058593.7,5539384.6
6,Unprocessed Cereals,561994.9,958949.7,610516.0,570810.1,572421.7,588247.4,523501.4,1118449.6,1065206.0,1082728.7,750655.0,701360.0,496589.8,899916.6,914234.4,1108800.3,924803.0,725122.0
7,Vegetables,34012893.3,60064931.2,37418975.9,34796375.2,34892755.4,35774348.6,31494455.7,69179920.5,66027921.6,66704460.4,45822504.9,43262398.1,29731686.4,55402914.1,56311534.7,69774330.5,57774666.7,44861516.7


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.41,0.05,0.01,0.01,0.03,0.60,0.55,0.56,0.23,0.19,0.41,0.42,0.63,0.44,0.24
1,Fats and Oils,0.55,0.07,0.02,0.02,0.04,0.79,0.73,0.74,0.30,0.25,0.54,0.56,0.84,0.58,0.32
2,Fruits,0.55,0.07,0.02,0.02,0.04,0.79,0.73,0.74,0.30,0.25,0.54,0.56,0.85,0.59,0.32
3,Meat and Prep,0.62,0.08,0.02,0.02,0.04,0.89,0.82,0.83,0.34,0.28,0.61,0.63,0.95,0.67,0.36
4,Processed Cereals,0.47,0.06,0.01,0.01,0.03,0.69,0.63,0.65,0.26,0.21,0.47,0.49,0.72,0.51,0.27
5,Sugar and Honey,0.47,0.06,0.01,0.01,0.03,0.70,0.63,0.65,0.26,0.21,0.47,0.48,0.71,0.50,0.27
6,Unprocessed Cereals,1.83,0.22,0.04,0.05,0.12,2.75,2.50,2.58,1.05,0.82,1.86,1.93,2.83,1.98,1.06
7,Vegetables,0.98,0.13,0.03,0.03,0.07,1.42,1.30,1.32,0.54,0.44,0.97,1.00,1.51,1.05,0.57
